In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/a udesa/7. septimo semestre/nlp/TP NLP 2026/notebooks')
print(os.getcwd())

!pip install -q datasets pandas

Mounted at /content/drive
/content/drive/MyDrive/a udesa/7. septimo semestre/nlp/TP NLP 2026/notebooks


In [ ]:
from datasets import load_dataset
import pandas as pd

LENGTH = "medium"

ds = load_dataset("avduarte333/BookTection", split="train")
df = ds.to_pandas()
df = df[df["Length"] == LENGTH].reset_index(drop=True)

# Limpiar filas rotas (#NAME? aparece en algunas celdas del CSV original)
opt_cols = ["Example_A", "Example_B", "Example_C", "Example_D"]
def ok(r): return all(isinstance(r[c], str) and r[c].strip() not in ("", "#NAME?") for c in opt_cols)
df = df[df.apply(ok, axis=1)].reset_index(drop=True)

print(f"{df['ID'].nunique()} books, {len(df)} pasajes (length={LENGTH})")
print("Members (1):", df[df.Label==1].ID.nunique(), "| Non-members (0):", df[df.Label==0].ID.nunique())
df.head(2)

165 books, 5494 pasajes (length=medium)
Members (1): 105 | Non-members (0): 60


,ID,Example_A,Example_B,Example_C,Example_D,Answer,Length,Label
0,1984_-_George_Orwell,"O'Brien had sat down beside the bed, so that h...",O'Brien positioned himself next to Winston's b...,O'Brien sat on the bed next to Winston so thei...,O'Brien took a seat on the bed next to Winston...,A,medium,1
1,1984_-_George_Orwell,The future belonged to the proles. And could h...,The future was owned by the proles. Winston wo...,The proles would inherit the future. Winston q...,The proles would come to rule the times ahead....,A,medium,1


In [ ]:
print(df.columns.tolist())

['ID', 'Example_A', 'Example_B', 'Example_C', 'Example_D', 'Answer', 'Length', 'Label']


In [ ]:
book_counts = (
    df.groupby("ID")
      .size()
      .sort_values(ascending=False)
)

book_counts.describe()

,0
count,165.000000
mean,33.296970
std,1.746859
min,19.000000
25%,33.000000
50%,34.000000
75%,34.000000
max,34.000000


In [ ]:
book_counts.head(20)

,0
ID,
A_Day_of_Fallen_Night_-_Samantha_Shannon,34
A_Game_of_Thrones_-_George_R._R._Martin,34
And_Then_There_Were_None_-_Agatha_Christie,34
After_Death_-_Dean_Koontz,34
A_Tale_of_Two_Cities_-_Charles_Dickens,34
Encyclopaedia_of_Faeries_-_Emily_Wildes,34
Eye_Of_The_Needle_-_Ken_Follet,34
Eclipse_-_Stephenie_Meyer,34
All_the_Light_We_Cannot_See_-_Anthony_Doerr,34


In [ ]:
book_label_counts = (
    df.groupby(["ID", "Label"])
      .size()
      .unstack(fill_value=0)
)

book_label_counts.columns = ["non_member", "member"]

book_label_counts.head()

,non_member,member
ID,,
1984_-_George_Orwell,0,33
A_Day_of_Fallen_Night_-_Samantha_Shannon,34,0
A_Game_of_Thrones_-_George_R._R._Martin,0,34
A_Living_Remedy_-_Nicole_Chung,33,0
A_Portrait_of_the_Artist_as_a_Young_Man_-_James_Joyce,0,33


In [ ]:
print(book_label_counts.describe())

       non_member      member
count  165.000000  165.000000
mean    12.054545   21.242424
std     16.014191   16.181950
min      0.000000    0.000000
25%      0.000000    0.000000
50%      0.000000   33.000000
75%     33.000000   34.000000
max     34.000000   34.000000


In [ ]:
RANDOM_STATE = 42
BOOKS_PER_CLASS = 10
CHUNKS_PER_BOOK = 10

book_counts = (
    df.groupby(["ID", "Label"])
      .size()
      .unstack(fill_value=0)
      .rename(columns={0: "non_member", 1: "member"})
)

display(book_counts.head())

Label,non_member,member
ID,,
1984_-_George_Orwell,0,33
A_Day_of_Fallen_Night_-_Samantha_Shannon,34,0
A_Game_of_Thrones_-_George_R._R._Martin,0,34
A_Living_Remedy_-_Nicole_Chung,33,0
A_Portrait_of_the_Artist_as_a_Young_Man_-_James_Joyce,0,33


In [ ]:
member_books = book_counts[
    (book_counts["member"] >= CHUNKS_PER_BOOK) &
    (book_counts["non_member"] == 0)
].copy()

print(len(member_books))
member_books.head()

105


Label,non_member,member
ID,,
1984_-_George_Orwell,0,33
A_Game_of_Thrones_-_George_R._R._Martin,0,34
A_Portrait_of_the_Artist_as_a_Young_Man_-_James_Joyce,0,33
A_Tale_of_Two_Cities_-_Charles_Dickens,0,34
Adventures_of_Huckleberry_Finn_-_Mark_Twain,0,33


In [ ]:
non_member_books = book_counts[
    (book_counts["non_member"] >= CHUNKS_PER_BOOK) &
    (book_counts["member"] == 0)
].copy()

print(len(non_member_books))
non_member_books.head()

60


Label,non_member,member
ID,,
A_Day_of_Fallen_Night_-_Samantha_Shannon,34,0
A_Living_Remedy_-_Nicole_Chung,33,0
A_Spell_of_Good_Things_-_Ayobami_Adebayo,31,0
After_Death_-_Dean_Koontz,34,0
Blowback_-_James_Patterson,34,0


In [ ]:
selected_member_books = (
    member_books
    .sample(BOOKS_PER_CLASS, random_state=RANDOM_STATE)
    .index
    .tolist()
)

selected_non_member_books = (
    non_member_books
    .sample(BOOKS_PER_CLASS, random_state=RANDOM_STATE)
    .index
    .tolist()
)

print("=== MEMBER BOOKS ===")
for b in selected_member_books:
    print(b)

print()

print("=== NON MEMBER BOOKS ===")
for b in selected_non_member_books:
    print(b)

=== MEMBER BOOKS ===
Harry_Potter_and_the_Chamber_of_Secrets_-_JK_Rowling
The_Alchemist_-_Paulo_Coelho
The_Age_of_Innocence_-_Edith_Wharton
Oliver_Twist_-_Charles_Dickens
Lord_of_the_Rings_The_Fellowship_of_the_Ring_-_J_R_R_Tolkien
The_Silmarillion_-_J._R._R._Tolkien
Wuthering_Heights_-_Emily_Bronte
Matilda_-_Roald_Dahl
Bartleby_the_Scrivener_A_Story_of_Wall_Street_-_Herman_Melville
1984_-_George_Orwell

=== NON MEMBER BOOKS ===
A_Day_of_Fallen_Night_-_Samantha_Shannon
Business_or_Pleasure_-_Rachel_Lynn_Solomon
The_Foxglove_King_-_Hannah_Whitten
The_Spectacular_-_Fiona_Davis
Hedge_-_Jane_Delury
Unfortunately_Yours_-_Tessa_Bailey
The_Darkness_Before_Them_-_Matthew_Ward
The_Wildest_Sun_A_Novel_-_Asha_Lemmie
Happy_Place_-_Emily_Henry
What_Lies_in_the_Woods_-_Kate_Alice_Marshall


In [ ]:
rows = []

for book in selected_member_books:
    sample = (
        df[(df.ID == book) & (df.Label == 1)]
        .sample(CHUNKS_PER_BOOK, random_state=RANDOM_STATE)
    )
    rows.append(sample)

for book in selected_non_member_books:
    sample = (
        df[(df.ID == book) & (df.Label == 0)]
        .sample(CHUNKS_PER_BOOK, random_state=RANDOM_STATE)
    )
    rows.append(sample)

evaluation_df = (
    pd.concat(rows)
      .sample(frac=1, random_state=RANDOM_STATE)
      .reset_index(drop=True)
)

In [ ]:
def get_correct_text(row):
    return row[f"Example_{row['Answer']}"]

evaluation_df["text"] = evaluation_df.apply(get_correct_text, axis=1)

In [ ]:
evaluation_df["sample_id"] = [
    f"BT_{i:04d}" for i in range(len(evaluation_df))
]

In [ ]:
evaluation_dataset = evaluation_df[
    [
        "sample_id",
        "ID",
        "Label",
        "Length",
        "Answer",
        "text",
        "Example_A",
        "Example_B",
        "Example_C",
        "Example_D"
    ]
].rename(columns={
    "ID": "book_id",
    "Label": "label",
    "Length": "length",
    "Answer": "answer"
})

In [ ]:
print("Ejemplos:", len(evaluation_dataset))
print("Libros:", evaluation_dataset["book_id"].nunique())

print("\nLabels")
print(evaluation_dataset["label"].value_counts())

print("\nChunks por libro")
print(evaluation_dataset.groupby("book_id").size())

Ejemplos: 200
Libros: 20

Labels
label
1    100
0    100
Name: count, dtype: int64

Chunks por libro
book_id
1984_-_George_Orwell                                               10
A_Day_of_Fallen_Night_-_Samantha_Shannon                           10
Bartleby_the_Scrivener_A_Story_of_Wall_Street_-_Herman_Melville    10
Business_or_Pleasure_-_Rachel_Lynn_Solomon                         10
Happy_Place_-_Emily_Henry                                          10
Harry_Potter_and_the_Chamber_of_Secrets_-_JK_Rowling               10
Hedge_-_Jane_Delury                                                10
Lord_of_the_Rings_The_Fellowship_of_the_Ring_-_J_R_R_Tolkien       10
Matilda_-_Roald_Dahl                                               10
Oliver_Twist_-_Charles_Dickens                                     10
The_Age_of_Innocence_-_Edith_Wharton                               10
The_Alchemist_-_Paulo_Coelho                                       10
The_Darkness_Before_Them_-_Matthew_Ward            

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/a udesa/7. septimo semestre/nlp/TP NLP 2026/datasets/evaluation"

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "booktection_medium_eval_20books_10chunks.csv"
)

evaluation_dataset.to_csv(OUTPUT_PATH, index=False)

print("Guardado en:")
print(OUTPUT_PATH)

Guardado en:
/content/drive/MyDrive/a udesa/7. septimo semestre/nlp/TP NLP 2026/datasets/evaluation/booktection_medium_eval_20books_10chunks.csv


In [ ]:
loaded = pd.read_csv(OUTPUT_PATH)

print(loaded.shape)
loaded.head()

(200, 10)


,sample_id,book_id,label,length,answer,text,Example_A,Example_B,Example_C,Example_D
0,BT_0000,1984_-_George_Orwell,1,medium,A,Since he was arrested he had not been fed. He ...,Since he was arrested he had not been fed. He ...,He had not eaten anything since being arrested...,"Since his arrest, he had been given nothing to...",He had been given no food since being arrested...
1,BT_0001,The_Alchemist_-_Paulo_Coelho,1,medium,A,"“And I know the Soul of the World, because we ...","“And I know the Soul of the World, because we ...",The essence of the universe has confided in me...,I'm privy to the psyche of the cosmos from our...,The spirit of the world has confided in me dur...
2,BT_0002,Oliver_Twist_-_Charles_Dickens,1,medium,A,"Master Bates followed, with a thoughtful count...","Master Bates followed, with a thoughtful count...","Master Bates trailed behind, looking pensive. ...","Master Bates came after, appearing thoughtful....","Master Bates followed behind, looking contempl..."
3,BT_0003,Unfortunately_Yours_-_Tessa_Bailey,0,medium,A,But August continues to try and fail and try a...,But August continues to try and fail and try a...,"However, August persists in attempting and fai...",But August keeps trying and failing and trying...,"However, August continues attempting and being..."
4,BT_0004,The_Foxglove_King_-_Hannah_Whitten,0,medium,A,"It creaked, but if the sound woke anyone insid...","It creaked, but if the sound woke anyone insid...","The door made noise when opened, but no one in...","The door creaked open, but didn't seem to dist...",The door squeaked open but didn't appear to wa...
